# 04 2.5D Qwen Evaluation

Inputs:

- output dataset from notebook `01_build_25d_cache.ipynb`
- output checkpoint from notebook `03_25d_qwen_sft_train.ipynb`
- same Qwen base model used in training

Minimal Q1 target: ROUGE-L >= 0.30, BLEU-4 >= 0.08, clinical keyword F1 >= 0.30,
SUV consistency >= 0.60 for a strong run. Smoke runs are diagnostic only.

In [ ]:
from pathlib import Path
import os, sys, subprocess, re, json, warnings, gc
warnings.filterwarnings("ignore")
os.environ.setdefault("PYDEVD_DISABLE_FILE_VALIDATION", "1")

MIN_PACKAGES = {
    "transformers": "4.51.0",
    "accelerate": "0.34.0",
    "peft": "0.12.0",
    "bitsandbytes": "0.46.1",
    "qwen-vl-utils": "0.0.8",
    "pillow": "10.0.0",
}

def version_tuple(v):
    nums = re.findall(r"\d+", str(v))[:4]
    return tuple(int(x) for x in nums) if nums else (0,)

def installed_version(package_name):
    try:
        from importlib import metadata
        return metadata.version(package_name)
    except Exception:
        return None

def ensure_package(package_name, min_version):
    current = installed_version(package_name)
    if current is not None and version_tuple(current) >= version_tuple(min_version):
        print(f"{package_name}: {current} OK")
        return
    spec = f"{package_name}>={min_version}"
    print(f"Installing {spec} (current={current})")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", spec])

for pkg, min_ver in MIN_PACKAGES.items():
    ensure_package(pkg, min_ver)

import numpy as np
import pandas as pd
from PIL import Image
import torch
from IPython.display import display

EVAL_SPLITS = ["validation", "test"]
RUN_MODE = os.environ.get("RUN_MODE", "smoke").lower()
SMOKE_MODE = RUN_MODE == "smoke"
MAX_EVAL_SAMPLES_PER_SPLIT = 16 if SMOKE_MODE else None
MODEL_ID_OR_PATH = os.environ.get("QWEN_MODEL_PATH", "Qwen/Qwen2.5-VL-3B-Instruct")
OUTPUT_ROOT = Path("/kaggle/working/vimedpet_q1_outputs") if Path("/kaggle/working").exists() else Path("vimedpet_q1_outputs")
EVAL_DIR = OUTPUT_ROOT / "04_qwen_eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_MODE:", RUN_MODE, "MAX_EVAL_SAMPLES_PER_SPLIT:", MAX_EVAL_SAMPLES_PER_SPLIT)
print("CUDA available:", torch.cuda.is_available(), "GPU count:", torch.cuda.device_count())


In [ ]:
import re, math, json
from collections import Counter

KEYWORDS = ["hạch", "phổi", "xương", "gan", "thận", "lách", "đại tràng", "SUV", "FDG", "tổn thương", "viêm", "PET/CT"]

def normalize_text(x):
    return re.sub(r"\s+", " ", str(x or "").strip())

def lcs_len(a, b):
    dp = [0] * (len(b) + 1)
    for ca in a:
        prev = 0
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev + 1 if ca == cb else max(dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

def rouge_l_f1(pred, ref):
    pred_toks = normalize_text(pred).split()
    ref_toks = normalize_text(ref).split()
    if not pred_toks or not ref_toks:
        return 0.0
    lcs = lcs_len(pred_toks, ref_toks)
    p = lcs / len(pred_toks)
    r = lcs / len(ref_toks)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def bleu4_simple(pred, ref):
    pred_toks = normalize_text(pred).split()
    ref_toks = normalize_text(ref).split()
    if len(pred_toks) < 4 or len(ref_toks) < 4:
        return 0.0
    scores = []
    for n in range(1, 5):
        pred_ngrams = Counter(tuple(pred_toks[i:i+n]) for i in range(len(pred_toks)-n+1))
        ref_ngrams = Counter(tuple(ref_toks[i:i+n]) for i in range(len(ref_toks)-n+1))
        overlap = sum((pred_ngrams & ref_ngrams).values())
        total = max(1, sum(pred_ngrams.values()))
        scores.append((overlap + 1) / (total + 1))
    bp = min(1.0, math.exp(1 - len(ref_toks) / max(1, len(pred_toks))))
    return float(bp * math.exp(sum(math.log(s) for s in scores) / 4))

def keyword_f1(pred, ref, keywords=KEYWORDS):
    pred_set = {kw.lower() for kw in keywords if kw.lower() in normalize_text(pred).lower()}
    ref_set = {kw.lower() for kw in keywords if kw.lower() in normalize_text(ref).lower()}
    if not pred_set and not ref_set:
        return 1.0
    tp = len(pred_set & ref_set)
    p = tp / max(1, len(pred_set))
    r = tp / max(1, len(ref_set))
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def extract_suv_values(text):
    values = []
    for m in re.finditer(r"SUV(?:max)?\s*[:=]?\s*([0-9]+(?:[\\.,][0-9]+)?)", str(text), flags=re.I):
        try:
            values.append(float(m.group(1).replace(",", ".")))
        except Exception:
            pass
    return values

def suv_tolerance_accuracy(pred, ref, tol=0.5):
    pv = extract_suv_values(pred)
    rv = extract_suv_values(ref)
    if not pv and not rv:
        return 1.0
    if not pv or not rv:
        return 0.0
    used = set()
    correct = 0
    for x in pv:
        candidates = [(abs(x - y), j) for j, y in enumerate(rv) if j not in used]
        if not candidates:
            continue
        err, j = min(candidates)
        if err <= tol:
            correct += 1
            used.add(j)
    return correct / max(len(rv), len(pv), 1)

def section_validity(text):
    t = normalize_text(text).lower()
    return int(("finding" in t or "mô tả" in t or "kết quả" in t) and ("impression" in t or "kết luận" in t))

def evaluate_prediction_frame(df, pred_col="prediction", ref_col="target_text"):
    rows = []
    for _, row in df.iterrows():
        pred, ref = row[pred_col], row[ref_col]
        rows.append({
            "rouge_l": rouge_l_f1(pred, ref),
            "bleu4": bleu4_simple(pred, ref),
            "keyword_f1": keyword_f1(pred, ref),
            "suv_acc_0_5": suv_tolerance_accuracy(pred, ref, tol=0.5),
            "section_valid": section_validity(pred),
        })
    out = pd.DataFrame(rows)
    return out.mean(numeric_only=True).to_dict()

In [ ]:
def locate(name):
    roots = list(Path("/kaggle/input").rglob(name)) if Path("/kaggle/input").exists() else []
    roots += list(Path.cwd().rglob(name))
    if not roots:
        raise FileNotFoundError(name)
    return sorted(roots, key=lambda p: len(str(p)))[0]

meta_path = locate("q1_cache_meta.json")
CACHE_DIR = meta_path.parent
meta = json.loads(meta_path.read_text())
clean = pd.read_csv(CACHE_DIR / "q1_clean_manifest_for_cache.csv")
cache_index = pd.read_csv(CACHE_DIR / "q1_cache_index.csv")
clean = clean.drop(columns=["row_index"], errors="ignore").merge(
    cache_index[cache_index.cache_ok][["case_id", "row_index"]], on="case_id", how="inner"
)

target_col = meta.get("target_column", "target_text")
if target_col in clean.columns:
    clean["base_report_text"] = clean[target_col].fillna("").astype(str)
elif "report_text_clean" in clean.columns:
    clean["base_report_text"] = clean["report_text_clean"].fillna("").astype(str)
else:
    clean["base_report_text"] = clean["target_text"].fillna("").astype(str)

def build_eval_target(row):
    findings = str(row.get("findings", "") or "").strip()
    if not findings:
        findings = str(row.get("base_report_text", "") or "").strip()
    return "Findings:\n" + findings

clean["target_text"] = clean.apply(build_eval_target, axis=1)
arr = np.memmap(
    CACHE_DIR / meta["mmap_path"],
    dtype=np.uint8,
    mode="r",
    shape=(meta["n"], meta["channels"], meta["image_size"], meta["image_size"]),
)
adapter_path = locate("adapter_config.json").parent
print("CACHE_DIR", CACHE_DIR)
print("adapter_path", adapter_path)

def row_to_rgb(row_index):
    x = arr[int(row_index)]
    return Image.fromarray(np.stack([x[1], x[4], x[3]], axis=-1), mode="RGB")


In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as QwenModel
except Exception:
    from transformers import AutoModelForVision2Seq as QwenModel
from peft import PeftModel

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 04 requires GPU. In Kaggle Settings, set Accelerator = GPU T4 x2.")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID_OR_PATH, trust_remote_code=True, use_fast=False)
if hasattr(processor, "tokenizer"):
    processor.tokenizer.padding_side = "right"
max_memory = {i: "14GiB" for i in range(torch.cuda.device_count())}
max_memory["cpu"] = "32GiB"
base = QwenModel.from_pretrained(
    MODEL_ID_OR_PATH,
    quantization_config=bnb,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)
model = PeftModel.from_pretrained(base, adapter_path)
model.eval()
print("device map:", getattr(model, "hf_device_map", None))


In [ ]:
PROMPT = (
    "Generate the available Vietnamese PET/CT report content from the image. "
    "Use the heading Findings. Do not invent an Impression section because it is not present in the source JSON."
)

def first_model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

pred_rows = []
for split in EVAL_SPLITS:
    df = clean[clean.clean_split == split].copy()
    if MAX_EVAL_SAMPLES_PER_SPLIT:
        df = df.sample(min(MAX_EVAL_SAMPLES_PER_SPLIT, len(df)), random_state=391)
    for _, row in df.iterrows():
        image = row_to_rgb(row.row_index)
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors="pt")
        device = first_model_device(model)
        inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=220, do_sample=False)
        gen = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
        pred_rows.append({"split": split, "case_id": row.case_id, "prediction": gen, "target_text": row.target_text})

pred_df = pd.DataFrame(pred_rows)
pred_df.to_csv(EVAL_DIR / "qwen_predictions.csv", index=False, encoding="utf-8-sig")
metrics = evaluate_prediction_frame(pred_df)
(EVAL_DIR / "qwen_eval_metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(metrics, indent=2, ensure_ascii=False))
display(pred_df.head(5))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
